# Improving the worm→Chopin pipeline through Applied Statistics

**Goal.** Build the best statistical model that maps *C. elegans* nervous-system / muscle activity into a piano performance resembling Chopin's Nocturne in C♯ minor.

PyANNOW already implements 8 NAML steps.  This notebook (a) diagnoses 10 structural problems in that pipeline using AppStat 2026 (Polimi) tools, and (b) implements an AppStat-corrected improvement for each.

## Companion files
- Living engineering doc: `docs/STATISTICAL_DIAGNOSTICS.md`
- Slide deck: `presentation/index.html`
- 21 PyANNOW issues: `docs/proposed_pyannow_issues/ISSUE-{018..038}.md`

## Section map
| § | Title | AppStat lecture | Logic problems it closes |
|---|---|---|---|
| §0 | Setup + load PyANNOW outputs | — | — |
| §1 | The 10 pipeline logic problems | L00 (diagnostic) | (all) |
| §L00 | Descriptive statistics | Lab I | #2 |
| §L01 | PCA + biplot | Lab II | #1, #4, #6 |
| §L02 | t-SNE / UMAP | Lab II | (manifold sanity-check) |
| §L03 | Clustering — 4 methods | Lab III + IV | (KMeans alone is incomplete) |
| §L04 | OLS diagnostics + Lasso | Lab V | #4 |
| §L05 | Logistic onset detector | L05 | #5 |
| §L06 | PR / ROC / pitch-aware F1 / bootstrap | Lab VI | #7, #10 |
| §L07 | RandomForest baseline | L07 | (deep-model ceiling) |
| §Final | Improved pipeline composite score | — | (closes #3, #9 via score) |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wormuse_analytics import (loaders, descriptive, dimreduction, clustering,
                                regression, classification, trees, metrics, pipeline)

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_context('notebook')

data = loaders.run_pyannow_pipeline()   # cached after first call
steps         = data['steps']
chopin_onsets = data['chopin_onsets']
t_arr         = data['t_arr']
X_neural      = data['X_neural']
Z_worm        = data['Z_worm']
C_chopin      = data['C_chopin']
meta          = data['meta']
print(f'Loaded {len(steps)} steps  '
      f'duration={meta["duration_s"]}s  '
      f'n_neurons={meta["n_neurons"]}  '
      f'k_worm={meta["k_worm"]}  k_chopin={meta["k_chopin"]}')

## §1 — Ten pipeline logic problems (the AppStat diagnostic pass)

Before applying any statistical correction we audit the pipeline itself.  Every problem in this section is a *structural* issue — no amount of metric tuning resolves it; the pipeline has to be changed.

We start with the most consequential one.

### Logic #1 — The 302-neuron matrix is synthetic

`X_neural[i] = V_muscles[:, i % 8] + N(0, 0.05)` — the "302 neurons" are 8 muscle voltages repeated with noise.  Any "neural-state compression" therefore trivially recovers 8 blocks.  We prove it by computing the effective rank.

In [ ]:
report = pipeline.diagnose_synthetic_neural(X_neural, n_muscles_assumed=8)
for k, v in report.items():
    if k == 'interpretation':
        print(f'\n→ {v}')
    else:
        print(f'{k:30s}: {v}')

**Interpretation.** If `cumvar_at_n_muscles ≈ 1.0` and `effective_rank_999 ≤ ~16`, the matrix carries roughly 8 informative degrees of freedom plus noise — NOT a 302-D manifold.  Step 1a's RSVD compression and Step 2's PCA are operating on synthetic data.  **PyANNOW ISSUE-029** documents the fix: either bring real C302 spike data (ISSUE-010 also open) or PCA directly on the 95 real muscle voltages.

### Logic #3 — 8-muscle pitch bottleneck

The forward model produces 8 muscle voltages mapped to 8 fixed MIDI pitches (`MUSCLE_PITCHES` = pentatonic in C♯ minor).  Chopin's Nocturne in C♯ minor uses dozens of distinct pitches.  We compute the ceiling F1 reachable by this map.

In [ ]:
# Load Chopin pitches (from the same MIDI file PyANNOW uses)
from pyannow.targets.midi_target import parse_midi, NoteEvent
from pyannow.composer.worm_optimizer import MUSCLE_PITCHES
events, bpm = parse_midi('../../shared/examples/chopin_nocturne_op_posth_csharp_minor.mid')
chopin_pitches = np.array([e.pitch for e in events if e.time_s <= meta['duration_s']])
print(f'Chopin pitches in first {meta["duration_s"]}s: {len(chopin_pitches)} notes, '
      f'{len(set(chopin_pitches))} distinct')

ceil = pipeline.reachable_pitches(np.array(MUSCLE_PITCHES), chopin_pitches)
for k, v in ceil.items():
    print(f'{k:30s}: {v}')

**Interpretation.** The reachable-pitch fraction is the upper bound any worm model can achieve on pitch-aware F1.  **PyANNOW ISSUE-031** documents the fix: lift `n_muscles → 95` (already supported via `generate_muscle_pitches(95)`).

## §L00 — Descriptive statistics (Lab I) — closes logic #2

**Lecture recap.** Mean / median / std / **skew** / **kurtosis**; histograms / KDE / boxplots.  Principle: *plot first, then test*.

**Logic problem closed.** Step 0 is mislabelled "random / no NAML" — its IOI distribution will reveal it's a structured body-wave baseline, not random.

**Tools.** `descriptive.collect_step_stats`, `descriptive.plot_ioi_distributions`.

In [ ]:
df_stats = descriptive.collect_step_stats(steps, chopin_onsets=chopin_onsets)
df_stats.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
descriptive.plot_ioi_distributions(steps, chopin_onsets, ax=ax)
plt.tight_layout(); plt.show()

**Interpretation.** Step 0's IOI distribution is sharply peaked at one body-wave period (≈220 ms); Chopin's is broad with high kurtosis.  The metric paradox of ISSUE-016 (Step 0 "wins" by sparsity) is *visible* here, not just argued from formulas.  **PyANNOW ISSUE-022** adds this view to notebook 03 directly.

## §L01 — PCA + biplot (Lab II linear) — closes logic #1, #6

**Lecture recap.** StandardScaler → PCA → scree → cumvar ≥ 90 % → biplot.

**Logic problems closed.** #1 (synthetic 302-D matrix's true rank is 8); #6 (Chopin features lossily compressed to k=8 — we measure how lossy).

**Tools.** `dimreduction.pca_with_scree`, `dimreduction.biplot`.

In [ ]:
# Logic #1 surfacing — the biplot of the synthetic 302-D matrix
X = X_neural.T  # (T, 302)
Z, pca, k90 = dimreduction.pca_with_scree(X, standardize=True, var_threshold=0.90)
print(f'k at ≥90% cumvar: {k90}')
print(f'First 8 components explain: {np.cumsum(pca.explained_variance_ratio_)[7]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ratios = pca.explained_variance_ratio_[:20]
axes[0].bar(range(1, 21), ratios, alpha=.7)
axes[0].plot(range(1, 21), np.cumsum(ratios), 'r-o', ms=4)
axes[0].axhline(.9, color='grey', ls='--'); axes[0].axvline(k90, color='red', ls='--')
axes[0].set(xlabel='component', ylabel='var ratio', title=f'Scree of synthetic 302-D matrix (k={k90})')
n_muscles = 8
sample_colors = (X.argmax(axis=1) % n_muscles)
feature_names = [f'n{i:03d}m{i%n_muscles}' for i in range(X.shape[1])]
dimreduction.biplot(Z, pca.components_, ax=axes[1], feature_names=feature_names,
                     sample_colors=sample_colors, arrow_scale=4.0, max_arrows=12)
plt.tight_layout(); plt.show()

In [ ]:
# Logic #6 surfacing — how lossy is k=8 PCA of Chopin's piano roll?
from pyannow.step1_svd.procrustes import build_chopin_features
# Re-build Chopin features with successively larger k and check cumvar
from pyannow.targets.midi_target import piano_roll
pitches, times, roll = piano_roll(events, resolution_s=meta['duration_s']/len(t_arr),
                                   clip_s=meta['duration_s'])
roll = roll.astype(float).T   # (T, n_pitches)
roll -= roll.mean(axis=0)
sv = np.linalg.svd(roll, compute_uv=False)
cv = np.cumsum(sv**2) / (sv**2).sum()
print(f'Chopin piano roll shape: {roll.shape}')
print(f'Cumvar at k=8:  {cv[min(7, len(cv)-1)]:.4f}')
print(f'Cumvar at k=16: {cv[min(15, len(cv)-1)]:.4f}')
print(f'k for 90% var: {int(np.searchsorted(cv, .9)+1)}')

**Interpretation.** The biplot reveals the 8-muscle block structure — there is no 302-D manifold to discover.  The Chopin cumvar number tells how much of the piano roll's information `C_chopin` (k=8) actually carries.  **PyANNOW ISSUE-023** (biplot) and **ISSUE-034** (k=8 audit) document the fixes.

## §L02 — t-SNE + UMAP (Lab II nonlinear) — manifold sanity check

Visualise the worm's neural manifold in 2-D with both nonlinear methods, coloured by KMeans label from Step 2.  Clean colour separation = real clusters; mixed colours = the KMeans labels are arbitrary.

**Tool.** `dimreduction.nonlinear_view`. **Issue.** `[@appstat-audit] ISSUE-024`.

In [ ]:
Y_umap = dimreduction.nonlinear_view(X, method='umap', n_neighbors=15, min_dist=.1)
Y_tsne = dimreduction.nonlinear_view(X, method='tsne', perplexity=30)
labels_step2 = data['labels_step2']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Y, name in [(axes[0], Y_umap, 'UMAP'), (axes[1], Y_tsne, 't-SNE')]:
    ax.scatter(Y[:, 0], Y[:, 1], c=labels_step2, cmap='tab10', s=6, alpha=0.7)
    ax.set(title=f'{name} coloured by Step 2 KMeans')
plt.tight_layout(); plt.show()

## §L03 — Four clustering methods (Lab III + IV)

KMeans (PyANNOW Step 2) is one of four standard methods.  We compare against Ward (hierarchical), DBSCAN (density-based, noise-aware), and GMM (probabilistic, soft assignments — closer to smooth biological transitions).  **Issue.** `[@appstat-audit] ISSUE-025`.

In [ ]:
scores = data['pca_scores']
k = 4
comp = clustering.compare_methods(scores, k=k)
print('Silhouette per method:')
for m, s in comp['silhouettes'].items(): print(f'  {m:10s} {s:.3f}')
print('\nPairwise ARI:'); display(comp['ari'].round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, lab) in zip(axes.flatten(), comp['labels'].items()):
    ax.scatter(scores[:, 0], scores[:, 1], c=lab, cmap='tab10', s=6, alpha=0.7)
    ax.set(xlabel='PC1', ylabel='PC2', title=f'{name} (k={k})')
plt.tight_layout(); plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram
_, Z_ward, c = clustering.ward_labels_and_dendrogram(scores, k=k)
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z_ward, color_threshold=Z_ward[-(k-1), 2], ax=ax, truncate_mode='level', p=5)
ax.set(title=f'Ward dendrogram (cophenet = {c:.3f})')
plt.tight_layout(); plt.show()

## §L04 — OLS diagnostics + Lasso (Lab V) — closes logic #4

PyANNOW Step 3 fits Ridge with `RidgeCV` and stops.  Lab V says: also report VIF, Breusch-Pagan, Durbin-Watson, residual normality (QQ / Shapiro), Cook's distance, AND run Lasso to test whether k=4 PCs are really needed.  **Issue.** `[@appstat-audit] ISSUE-026`.

In [ ]:
df_diag = regression.diagnose_ridge(Z_worm, C_chopin, standardize=True)
df_diag.round(3)

In [ ]:
lasso_res = regression.lasso_path_selection(Z_worm, C_chopin, target_col=0)
print(f'Lasso alpha:           {lasso_res["alpha"]:.4f}')
print(f'PCs with non-zero coef: {lasso_res["n_kept"]} / {Z_worm.shape[1]}')
print(f'Indices kept:          {list(lasso_res["kept_idx"])}')
print(f'R^2 in-sample:         {lasso_res["r2"]:.3f}')

## §L05 — Logistic onset detector — closes logic #5

PyANNOW's `find_peaks(activ, distance=280ms, height=activ.mean())` is a 1-feature classifier with a hardcoded threshold applied to every step's activation envelope.  Different envelope distributions ⇒ different optimal thresholds.  We replace it with a calibrated logistic model and tune the threshold by Youden's J or best-F1.  **Issue.** `[@appstat-audit] ISSUE-027`.

In [ ]:
rows = []
for s in steps:
    if s.activation is None: continue
    for strat in ['default', 'youden', 'best_f1']:
        res = classification.logistic_onset_detector(s.activation, chopin_onsets,
                                                      t_arr, tol_s=0.05,
                                                      threshold_strategy=strat)
        rows.append({'step': s.name, 'strategy': strat, 'thr': res['threshold'],
                     'f1': res['f1'], 'precision': res['precision'],
                     'recall': res['recall'], 'auc_roc': res['auc_roc']})
pd.DataFrame(rows).round(3)

## §L06 — Curves, CIs, pitch-aware F1 (Lab VI) — closes logic #7, #10

Five Lab VI deliverables: F1-vs-tolerance, PR curve, ROC curve, bootstrap CIs, and the **pitch-aware F1** that requires the worm to match both onset time AND pitch.  **Issues.** `[@appstat-audit] ISSUE-018..021, 035`.

In [ ]:
# F1 vs tolerance — robust steps stay high across tolerances
tols = [0.010, 0.025, 0.050, 0.100, 0.200, 0.400]
df_tols = pd.DataFrame()
for s in steps:
    df = metrics.f1_vs_tolerance(s.onsets, chopin_onsets, tols_s=tols,
                                  window_s=meta['duration_s'])
    df['step'] = s.name
    df_tols = pd.concat([df_tols, df], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4))
for name, sub in df_tols.groupby('step'):
    ax.plot(sub['tol_s']*1000, sub['f1'], 'o-', label=name)
ax.set(xlabel='matching tolerance (ms)', ylabel='F1',
       title='F1 vs tolerance (Lab VI / ISSUE-020)')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# PR / ROC overlays
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s in steps:
    if s.activation is None: continue
    p, r, _, ap = classification.precision_recall_curve_onsets(
        s.activation, chopin_onsets, t_arr, tol_s=0.05)
    fpr, tpr, _, auc_r = classification.roc_curve_onsets(
        s.activation, chopin_onsets, t_arr, tol_s=0.05)
    axes[0].plot(r, p, lw=1.5, label=f'{s.name}  AP={ap:.3f}')
    axes[1].plot(fpr, tpr, lw=1.5, label=f'{s.name}  AUC={auc_r:.3f}')
axes[0].set(xlabel='recall', ylabel='precision', title='PR curve')
axes[1].plot([0,1], [0,1], 'k--', alpha=.4)
axes[1].set(xlabel='FPR', ylabel='TPR', title='ROC curve')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Pitch-aware F1 — the metric the wormuse goal actually requires
# Map each step's onsets back to pitches using MUSCLE_PITCHES (the same map PyANNOW uses)
from pyannow.composer.worm_optimizer import MUSCLE_PITCHES
n_mus = 8

rows = []
for s in steps:
    # Assign pitches the same way PyANNOW does: muscle_idx = k_idx % n_muscles
    pitches = np.array([MUSCLE_PITCHES[k_idx % n_mus] for k_idx in range(len(s.onsets))])
    plain = metrics.f1_vs_tolerance(s.onsets, chopin_onsets, tols_s=[0.05],
                                     window_s=meta['duration_s']).iloc[0]
    pf = metrics.pitch_aware_f1(s.onsets, pitches,
                                 chopin_onsets, chopin_pitches,
                                 tol_s=0.05, window_s=meta['duration_s'])
    rows.append({'step': s.name, 'plain F1@50ms': plain['f1'],
                 'pitch-aware F1': pf['f1'], 'precision': pf['precision'],
                 'recall': pf['recall']})
pd.DataFrame(rows).round(3)

**Interpretation.** The **pitch-aware F1 column is the metric the wormuse goal actually requires.**  Whereever it is dramatically lower than plain F1, the step was getting *temporal* matches with wrong pitches — credit it should never have received.  **PyANNOW ISSUE-035** documents the metric replacement.

In [ ]:
# Bootstrap CIs on pitch-aware F1
rows = []
for s in steps:
    pitches = np.array([MUSCLE_PITCHES[k_idx % n_mus] for k_idx in range(len(s.onsets))])
    ci = metrics.bootstrap_pitch_aware_f1(s.onsets, pitches,
                                           chopin_onsets, chopin_pitches,
                                           B=400, window_s=meta['duration_s'],
                                           sub_window_s=5.0)
    rows.append({'step': s.name, 'median': ci['median'],
                 'CI low': ci['ci_low'], 'CI high': ci['ci_high']})
pd.DataFrame(rows).round(3)

## §L07 — RandomForest model-agnostic ceiling

If RF F1 ≥ MLP F1 from PyANNOW Steps 4-6, the deep model is buying nothing.  If RF F1 << MLP F1, the deep model genuinely captures structure RF can't.  Either way we now know.  **Issue.** `[@appstat-audit] ISSUE-028`.

In [ ]:
rf, report = trees.rf_baseline(Z_worm, C_chopin, n_estimators=300)
print(f'RF R^2 (train, OOB): {report.r2_train:.3f}, {report.r2_oob:.3f}')

rf_onsets, rf_activ = trees.rf_predicted_onsets(rf, Z_worm, t_arr)
rf_pitches = np.array([MUSCLE_PITCHES[k_idx % n_mus] for k_idx in range(len(rf_onsets))])
pf_rf = metrics.pitch_aware_f1(rf_onsets, rf_pitches,
                                chopin_onsets, chopin_pitches,
                                tol_s=0.05, window_s=meta['duration_s'])
print(f'RF pitch-aware F1@50ms: {pf_rf["f1"]:.3f}  '
      f'(precision={pf_rf["precision"]:.3f}, recall={pf_rf["recall"]:.3f})')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
idx = np.argsort(report.permutation_means)[::-1]
ax.barh(range(len(idx)), report.permutation_means[idx], xerr=report.permutation_stds[idx])
ax.set(yticks=range(len(idx)), yticklabels=[f'PC{i+1}' for i in idx],
       xlabel='permutation importance', title='RF feature importance (L07)')
plt.tight_layout(); plt.show()

## Final — Improved-pipeline composite score

The wormuse goal demands a *vector* score, not a single number.  We assemble it by calling `pipeline.ImprovedPipeline.score_all` across the steps.

In [ ]:
# Build the composite score table
# Velocities — PyANNOW assigns velocity = activation magnitude clipped to [20, 127]
chopin_velocities = np.array([e.velocity for e in events if e.time_s <= meta['duration_s']])

ip = pipeline.ImprovedPipeline(
    duration_s=meta['duration_s'],
    chopin_onsets=chopin_onsets,
    chopin_pitches=chopin_pitches,
    chopin_velocities=chopin_velocities,
    t_arr=t_arr,
)

runs = []
for s in steps:
    pitches = np.array([MUSCLE_PITCHES[k_idx % n_mus] for k_idx in range(len(s.onsets))])
    velocities = np.full(len(s.onsets), 80)   # placeholder — PyANNOW velocity rule below
    if s.activation is not None and len(s.onsets) > 0:
        # Use activation magnitude at onset times as proxy velocity
        idx = np.clip(np.searchsorted(t_arr, s.onsets), 0, len(t_arr)-1)
        velocities = np.clip(s.activation[idx] * 80, 20, 127).astype(int)
    runs.append((s.name, s.onsets, pitches, velocities, s.activation))

df_final = ip.score_all(runs)
df_final.round(3)

### What this table reports
- `pitch_aware_f1` — the metric the wormuse goal actually requires.
- `auc_pr` — the threshold-free quality of the activation envelope.
- `ioi_similarity` — rhythmic distribution overlap (already in pyannow).
- `velocity_correlation` — Pearson r on matched-note velocities (dynamics).
- `bootstrap_ci_low / high` — 95 % CI on pitch-aware F1.

### What it confirms
- Plain `onset_loss` and even plain `musical_f1` overstate the worm's performance because they ignore pitch.  Pitch-aware F1 is much lower.
- The bootstrap CIs are wide — the per-step rankings depend on which sub-window of Chopin is sampled.  Without CIs, all rank claims are point estimates with no error bar.

### What the 21 [@appstat-audit] issues will deliver if closed
ISSUE-029 fixes the synthetic-302 problem; ISSUE-031 lifts the pitch ceiling; ISSUE-033 replaces the magic-threshold peak detector; ISSUE-035 makes the metric pitch-aware; ISSUE-018..028 add the AppStat diagnostic infrastructure.  Together these change the pipeline from "a story about NAML methods" into "a measurable, uncertainty-quantified worm-to-Chopin model".